# MobileViTv2-0.75 baseline on ImageNet-100

Training MobileViTv2-0.75 from scratch as a size-matched comparison for CBA-MobileViT-S-lite.

**Note on schedule mismatch:** this run uses B=128, LR=4e-4 (~4x more optimizer steps/epoch than CBA-lite and XS at B=512). The comparison to CBA-lite is one-sided in v2-0.75's favour — the +3.48pp CBA-lite gap is understated, not overstated.

**Recipe:** 256x256, 50 epochs, AdamW decoupled WD, B=128, LR=4e-4, cosine + 5-epoch warmup, RandAugment(N=2, M=9), Mixup(0.2)/CutMix(1.0) p=0.5, drop_path=0.05, EMA decay=0.999, fp16 AMP.

In [1]:
import os, math, time, json, copy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms as T

import timm
from datasets import load_dataset
from tqdm.auto import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available"}')

PyTorch : 2.10.0+cu128
CUDA    : 12.8
GPU     : Tesla T4


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CFG = {
    'model_name'         : 'mobilevitv2_075',   # timm name; pretrained=False

    # training
    'epochs'             : 50,
    'batch_size'         : 128,        # smaller than CBA-lite/XS — see header note on schedule mismatch
    'num_workers'        : 8,
    'num_classes'        : 100,
    'img_size'           : 256,

    # optimizer
    'lr'                 : 4e-4,       # canonical DeiT/MobileViT LR at B=128
    'weight_decay'       : 0.05,
    'grad_clip'          : 1.0,

    'warmup_epochs'      : 5,

    'label_smoothing'    : 0.1,

    # augmentation
    'randaug_n'          : 2,
    'randaug_m'          : 9,
    'mixup_alpha'        : 0.2,
    'cutmix_alpha'       : 1.0,
    'mixup_prob'         : 0.5,
    'mixup_switch'       : 0.5,

    # regularization
    'drop_path_rate'     : 0.05,
    'ema_decay'          : 0.999,

    'device'             : device,

    # fp16 + GradScaler (run on earlier GPU without bf16 support)
    'amp'                : True,

    'save_dir'           : Path('/kaggle/working/checkpoints_control'),
    'run_name'           : 'mobilevitv2075_in100_fromscratch',
    'seed'               : 42,
}

CFG['save_dir'].mkdir(parents=True, exist_ok=True)
torch.manual_seed(CFG['seed'])
print(f'Training on: {device}')

Training on: cuda


In [3]:
_MEAN = (0.485, 0.456, 0.406)
_STD  = (0.229, 0.224, 0.225)


def _build_transforms(cfg):
    train_tf = T.Compose([
        T.RandomResizedCrop(cfg['img_size'], scale=(0.08, 1.0)),
        T.RandomHorizontalFlip(),
        T.RandAugment(num_ops=cfg['randaug_n'], magnitude=cfg['randaug_m']),
        T.ToTensor(),
        T.Normalize(_MEAN, _STD),
    ])
    val_tf = T.Compose([
        T.Resize(int(cfg['img_size'] / 0.875)),
        T.CenterCrop(cfg['img_size']),
        T.ToTensor(),
        T.Normalize(_MEAN, _STD),
    ])
    return train_tf, val_tf


class HFImageDataset(Dataset):
    def __init__(self, hf_split, transform):
        self.ds = hf_split
        self.tf = transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        return self.tf(item['image'].convert('RGB')), item['label']


print('Loading ImageNet-100 from HuggingFace (warming cache)...')
_warm = load_dataset('clane9/imagenet-100')
print(f"Train size : {len(_warm['train'])}")
print(f"Val size   : {len(_warm['validation'])}")
del _warm

Loading ImageNet-100 from HuggingFace (warming cache)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

data/train-00000-of-00017.parquet:   0%|          | 0.00/505M [00:00<?, ?B/s]

...

data/validation-00000-of-00001.parquet:   0%|          | 0.00/314M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/126689 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

Train size : 126689
Val size   : 5000


## Model: MobileViTv2-0.75 (random init via timm)

CPU-side sanity check verifying the model builds and produces the right output shape before training starts.

In [4]:
# sanity check before training: model builds with the correct output shape
_model_cpu = timm.create_model(
    CFG['model_name'],
    pretrained=False,
    num_classes=CFG['num_classes'],
    drop_path_rate=CFG['drop_path_rate'],
)
total_params = sum(p.numel() for p in _model_cpu.parameters())
print(f'Model               : {CFG["model_name"]} (random init, no pretrained weights)')
print(f'Total parameters    : {total_params:>12,}')

_model_cpu.eval()
with torch.no_grad():
    _dummy = torch.zeros(2, 3, CFG['img_size'], CFG['img_size'])
    _out   = _model_cpu(_dummy)
print(f'Forward pass: {tuple(_dummy.shape)} -> {tuple(_out.shape)}')
assert _out.shape == (2, CFG['num_classes']), \
    f'Expected (2, {CFG["num_classes"]}), got {_out.shape}'
print('Shape check passed')
del _model_cpu, _dummy, _out

Model               : mobilevitv2_075 (random init, no pretrained weights)
Total parameters    :    2,519,509
Forward pass: (2, 3, 256, 256) → (2, 100)
Shape check passed ✓


## GPU Training

Single-GPU loop with fp16 + GradScaler. Same architecture comparison as the XS baseline — only the model differs.

In [5]:
# applies Mixup or CutMix per batch (50/50 given switch_prob, with overall prob `prob`)
# returns (imgs, label_a, label_b, lambda) for the mixed CE loss
class MixupCutmix:
    def __init__(self, mixup_alpha=0.2, cutmix_alpha=1.0, prob=0.5, switch_prob=0.5):
        self.mixup_alpha = mixup_alpha
        self.cutmix_alpha = cutmix_alpha
        self.prob = prob
        self.switch_prob = switch_prob

    def __call__(self, imgs, labels):
        if torch.rand(1).item() >= self.prob:
            return imgs, labels, labels, 1.0
        idx = torch.randperm(imgs.size(0), device=imgs.device)
        if torch.rand(1).item() < self.switch_prob:
            lam = float(np.random.beta(self.cutmix_alpha, self.cutmix_alpha))
            H, W = imgs.shape[2], imgs.shape[3]
            cut_rat = math.sqrt(1.0 - lam)
            cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
            cx, cy = np.random.randint(W), np.random.randint(H)
            x1, y1 = max(0, cx - cut_w // 2), max(0, cy - cut_h // 2)
            x2, y2 = min(W, cx + cut_w // 2), min(H, cy + cut_h // 2)
            imgs[:, :, y1:y2, x1:x2] = imgs[idx, :, y1:y2, x1:x2]
            lam = 1.0 - ((x2 - x1) * (y2 - y1) / float(W * H))
        else:
            lam = float(np.random.beta(self.mixup_alpha, self.mixup_alpha))
            imgs = lam * imgs + (1.0 - lam) * imgs[idx]
        return imgs, labels, labels[idx], lam


# exponential moving average of model weights; BN buffers are copied not averaged
class ModelEMA:
    def __init__(self, model, decay=0.9995):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        d = self.decay
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.mul_(d).add_(msd[k].detach(), alpha=1.0 - d)
            else:
                v.copy_(msd[k])


# weight decay on conv/linear weights only, skips BN, LN and biases
def build_param_groups(model, weight_decay):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim <= 1 or n.endswith('.bias'):
            no_decay.append(p)
        else:
            decay.append(p)
    return [
        {'params': decay,    'weight_decay': weight_decay},
        {'params': no_decay, 'weight_decay': 0.0},
    ]


# one training epoch with Mixup/CutMix and EMA update
# accuracy returned as fraction [0,1] here — multiplied by 100 in the print loop
def train_one_epoch_gpu(model, loader, criterion, optimizer, scaler, mixup, ema, cfg, device, epoch):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    t0 = time.time()

    for images, labels in tqdm(loader, desc=f'Train {epoch+1:>3}', leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        images, y_a, y_b, lam = mixup(images, labels)

        optimizer.zero_grad()
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(images)
            loss   = lam * criterion(logits, y_a) + (1.0 - lam) * criterion(logits, y_b)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg['grad_clip'])
        scaler.step(optimizer)
        scaler.update()

        if ema is not None:
            ema.update(model)

        bs          = images.size(0)
        total_loss += loss.item() * bs
        correct    += (logits.argmax(1) == y_a).sum().item()
        total      += bs

    return total_loss / total, correct / total, time.time() - t0


@torch.no_grad()
def evaluate_gpu(model, loader, criterion, device):
    # returns (loss, top1_fraction, top5_fraction) in [0,1]
    model.eval()
    total_loss, c1, c5, total = 0.0, 0, 0, 0

    for images, labels in tqdm(loader, desc='  Val    ', leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(images)
            loss   = criterion(logits, labels)
        bs          = images.size(0)
        total_loss += loss.item() * bs
        c1         += (logits.argmax(1) == labels).sum().item()
        _, top5     = logits.topk(min(5, logits.size(1)), dim=1)
        c5         += (top5 == labels.unsqueeze(1)).any(1).sum().item()
        total      += bs

    return total_loss / total, c1 / total, c5 / total


def measure_throughput_gpu(model, input_size, device, batch_size=64, n_warmup=20, n_runs=100):
    model.eval()
    dummy = torch.randn(batch_size, 3, *input_size, device=device)

    for _ in range(n_warmup):
        with torch.autocast('cuda', dtype=torch.float16):
            _ = model(dummy)
    torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(n_runs):
        with torch.autocast('cuda', dtype=torch.float16):
            _ = model(dummy)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    return (batch_size * n_runs) / elapsed


def run_training_gpu(model_name, timm_name, cfg):
    # train from scratch on a single GPU, return history dict
    device = cfg['device']

    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        _alloc = torch.cuda.memory_allocated() / 1e9
        _resvd = torch.cuda.memory_reserved()  / 1e9
        print(f'GPU after cleanup: {_alloc:.2f} GB allocated, {_resvd:.2f} GB reserved')

    model = timm.create_model(
        timm_name,
        pretrained=False,
        num_classes=cfg['num_classes'],
        drop_path_rate=cfg['drop_path_rate'],
    )
    model = model.to(device)

    input_size = (cfg['img_size'], cfg['img_size'])
    n_params   = sum(p.numel() for p in model.parameters()) / 1e6

    print(f'\n{"=" * 60}')
    print(f'  Model : {model_name}  ({timm_name})  on {device}')
    print(f'{"=" * 60}')
    print(f'  Random init      : pretrained=False')
    print(f'  Input size       : {input_size}')
    print(f'  Params (total)   : {n_params:.2f}M')
    print(f'  Batch size       : {cfg["batch_size"]}')
    print(f'  Epochs           : {cfg["epochs"]}')

    train_tf, val_tf = _build_transforms(cfg)
    hf_dataset = load_dataset('clane9/imagenet-100')
    train_ds   = HFImageDataset(hf_dataset['train'],      train_tf)
    val_ds     = HFImageDataset(hf_dataset['validation'], val_tf)

    _pw = cfg['num_workers'] > 0
    train_loader = DataLoader(
        train_ds, batch_size=cfg['batch_size'], shuffle=True,
        num_workers=cfg['num_workers'], drop_last=True,
        persistent_workers=_pw, pin_memory=True,
        prefetch_factor=4 if _pw else None,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg['batch_size'] * 2, shuffle=False,
        num_workers=cfg['num_workers'], drop_last=False,
        persistent_workers=_pw, pin_memory=True,
        prefetch_factor=4 if _pw else None,
    )

    criterion = nn.CrossEntropyLoss(label_smoothing=cfg['label_smoothing'])
    optimizer = optim.AdamW(
        build_param_groups(model, cfg['weight_decay']),
        lr=cfg['lr'], betas=(0.9, 0.999),
    )
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.get('amp', True))
    mixup  = MixupCutmix(
        mixup_alpha=cfg['mixup_alpha'],
        cutmix_alpha=cfg['cutmix_alpha'],
        prob=cfg['mixup_prob'],
        switch_prob=cfg['mixup_switch'],
    )
    ema = ModelEMA(model, decay=cfg['ema_decay'])

    def _lr_lambda(ep):
        # linear warmup then cosine decay
        if ep < cfg['warmup_epochs']:
            return (ep + 1) / cfg['warmup_epochs']
        t = (ep - cfg['warmup_epochs']) / max(1, cfg['epochs'] - cfg['warmup_epochs'])
        return 0.5 * (1.0 + math.cos(math.pi * t))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, _lr_lambda)

    n_decay   = sum(p.numel() for p in optimizer.param_groups[0]['params'])
    n_nodecay = sum(p.numel() for p in optimizer.param_groups[1]['params'])
    print(f'  Optim            : AdamW lr={cfg["lr"]} wd={cfg["weight_decay"]} '
          f'(decay={n_decay:,}, no-decay={n_nodecay:,})')
    print(f'  Sched            : Cosine, warmup={cfg["warmup_epochs"]} ep')
    print(f'  Aug              : RandAug(N={cfg["randaug_n"]}, M={cfg["randaug_m"]}) + '
          f'Mixup({cfg["mixup_alpha"]})/CutMix({cfg["cutmix_alpha"]}) p={cfg["mixup_prob"]}')
    print(f'  Reg              : drop_path={cfg["drop_path_rate"]} EMA decay={cfg["ema_decay"]}')

    history = {'epoch': [], 'train_loss': [], 'train_acc': [],
               'val_loss': [], 'val_acc1': [], 'val_acc5': [],
               'ema_val_loss': [], 'ema_val_acc1': [], 'ema_val_acc5': [],
               'lr': []}
    best_acc   = 0.0
    best_ckpt  = cfg['save_dir'] / f"{cfg['run_name']}_best.pth"
    final_ckpt = cfg['save_dir'] / f"{cfg['run_name']}_final.pth"

    for epoch in range(cfg['epochs']):
        t_loss, t_acc, t_time = train_one_epoch_gpu(
            model, train_loader, criterion, optimizer, scaler, mixup, ema, cfg, device, epoch)
        v_loss,  v_acc1,  v_acc5  = evaluate_gpu(model,      val_loader, criterion, device)
        ev_loss, ev_acc1, ev_acc5 = evaluate_gpu(ema.module, val_loader, criterion, device)
        scheduler.step()

        cur_lr     = scheduler.get_last_lr()[0]
        epoch_best = max(v_acc1, ev_acc1)
        if epoch_best > best_acc:
            best_acc = epoch_best
            torch.save({
                'model_name'     : timm_name,
                'num_classes'    : cfg['num_classes'],
                'state_dict'     : model.state_dict(),
                'ema_state_dict' : ema.module.state_dict(),
                'epoch'          : epoch + 1,
                'val_acc1'       : v_acc1,
                'ema_val_acc1'   : ev_acc1,
            }, best_ckpt)

        history['epoch'].append(epoch + 1)
        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc1'].append(v_acc1)
        history['val_acc5'].append(v_acc5)
        history['ema_val_loss'].append(ev_loss)
        history['ema_val_acc1'].append(ev_acc1)
        history['ema_val_acc5'].append(ev_acc5)
        history['lr'].append(cur_lr)

        # accuracies from evaluate_gpu are fractions; multiply by 100 for the print
        print(f'E{epoch+1:>3}/{cfg["epochs"]}  '
              f'tr_loss={t_loss:.4f}  tr_acc={t_acc*100:5.2f}  '
              f'val={v_acc1*100:5.2f}/{v_acc5*100:5.2f}  '
              f'ema={ev_acc1*100:5.2f}/{ev_acc5*100:5.2f}  '
              f'best={best_acc*100:5.2f}  lr={cur_lr:.2e}  t={t_time:.0f}s')

    torch.save({
        'model_name'     : timm_name,
        'num_classes'    : cfg['num_classes'],
        'state_dict'     : model.state_dict(),
        'ema_state_dict' : ema.module.state_dict(),
        'epoch'          : cfg['epochs'],
        'val_acc1'       : v_acc1,
        'ema_val_acc1'   : ev_acc1,
    }, final_ckpt)

    throughput = measure_throughput_gpu(model, input_size, device)
    print(f'\n  Best val acc     : {best_acc*100:.2f}%  (live or EMA)')
    print(f'  Throughput       : {throughput:.0f} img/s')
    print(f'  Best ckpt        : {best_ckpt}')
    print(f'  Final ckpt       : {final_ckpt}')

    history['best_val_acc'] = best_acc
    history['throughput']   = throughput
    history['model_name']   = model_name
    history['input_size']   = list(input_size)
    history['params_M']     = n_params

    with open(cfg['save_dir'] / f"{cfg['run_name']}_history.json", 'w') as f:
        json.dump(history, f, indent=2)

    return history

In [6]:
history = run_training_gpu(CFG['model_name'], CFG['model_name'], CFG)

GPU after cleanup: 0.00 GB allocated, 0.00 GB reserved

  Model : mobilevitv2_075  (mobilevitv2_075)  on cuda
  Random init      : pretrained=False
  Input size       : (256, 256)
  Params (total)   : 2.52M
  Batch size       : 128
  Epochs           : 50


Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

  Optim            : AdamW lr=0.0004 wd=0.05 (decay=2,493,432, no-decay=26,077)
  Sched            : Cosine, warmup=5 ep
  Aug              : RandAug(N=2, M=9) + Mixup(0.2)/CutMix(1.0) p=0.5
  Reg              : drop_path=0.05 EMA decay=0.999


...

E 49/50  tr_loss=2.3321  tr_acc=52.37  val=78.04/94.38  ema=77.76/94.42  best=78.04  lr=4.87e-07  t=570s


Train  50:   0%|          | 0/989 [00:00<?, ?it/s]

  Val    :   0%|          | 0/20 [00:00<?, ?it/s]

  Val    :   0%|          | 0/20 [00:00<?, ?it/s]

E 50/50  tr_loss=2.2433  tr_acc=53.51  val=77.90/94.40  ema=77.86/94.38  best=78.04  lr=0.00e+00  t=570s

  Best val acc     : 78.04%  (live or EMA)
  Throughput       : 764 img/s
  Best ckpt        : /kaggle/working/checkpoints_control/mobilevitv2075_in100_fromscratch_best.pth
  Final ckpt       : /kaggle/working/checkpoints_control/mobilevitv2075_in100_fromscratch_final.pth


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'],   label='Train',   color='#2980b9', lw=1.5)
axes[0].plot(history['val_loss'],     label='Val',     color='#e74c3c', lw=1.5)
axes[0].plot(history['ema_val_loss'], label='Val EMA', color='#8e44ad', lw=1.5, ls='--')
axes[0].set(title='Loss', xlabel='Epoch', ylabel='Cross-entropy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot([a*100 for a in history['train_acc']],    label='Train top-1 (orig labels)', color='#2980b9', lw=1.5)
axes[1].plot([a*100 for a in history['val_acc1']],     label='Val top-1',                 color='#e74c3c', lw=1.5)
axes[1].plot([a*100 for a in history['ema_val_acc1']], label='EMA top-1',                 color='#8e44ad', lw=1.5)
axes[1].plot([a*100 for a in history['val_acc5']],     label='Val top-5',                 color='#27ae60', lw=1.5, ls='--')
axes[1].plot([a*100 for a in history['ema_val_acc5']], label='EMA top-5',                 color='#16a085', lw=1.5, ls='--')
axes[1].set(title='Accuracy', xlabel='Epoch', ylabel='Acc (%)')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

ax = axes[2]
ax.axis('off')
summary = [
    f'Model: {CFG["model_name"]} (random init)',
    f'Params: {history.get("params_M", 0):.2f}M',
    f'Epochs: {CFG["epochs"]}',
    f'Dataset: ImageNet-100 @ {CFG["img_size"]}',
    f'Hardware: GPU (fp16 AMP)',
    f'',
    f'Best Val Top-1:   {max(history["val_acc1"])*100:.2f}%',
    f'Best EMA Top-1:   {max(history["ema_val_acc1"])*100:.2f}%',
    f'Best Val Top-5:   {max(history["val_acc5"])*100:.2f}%',
    f'Best EMA Top-5:   {max(history["ema_val_acc5"])*100:.2f}%',
    f'Final Train Acc:  {history["train_acc"][-1]*100:.2f}%',
    f'Train-Val gap:    {(history["train_acc"][-1] - history["val_acc1"][-1])*100:+.2f}%',
    f'Throughput:       {history.get("throughput", 0):.0f} img/s',
]
ax.text(0.05, 0.95, '\n'.join(summary), transform=ax.transAxes,
        va='top', fontsize=11, family='monospace',
        bbox=dict(boxstyle='round', facecolor='#f8f8f8', alpha=0.8))
ax.set_title('Summary')

plt.suptitle(f'{CFG["model_name"]} (from scratch, GPU) — ImageNet-100', fontsize=13, y=1.01)
plt.tight_layout()
out_fig = CFG['save_dir'] / f"{CFG['run_name']}_results.png"
plt.savefig(out_fig, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved {out_fig}')

Saved /kaggle/working/checkpoints_control/mobilevitv2075_in100_fromscratch_results.png


---
## Loading the saved checkpoint

```python
import torch, timm

ckpt = torch.load('/kaggle/working/checkpoints_control/mobilevitv2075_in100_fromscratch_best.pth',
                  map_location='cpu')
model = timm.create_model(ckpt['model_name'], pretrained=False, num_classes=ckpt['num_classes'])
model.load_state_dict(ckpt['ema_state_dict'])  # use ema_state_dict for inference
model.eval()
```

Two checkpoints are saved: `_best.pth` (highest live-or-EMA val top-1 during training) and `_final.pth` (last epoch). Use `_best.pth` for inference, `_final.pth` if continuing training.